In [0]:
from pyspark.sql.functions import *

In [0]:
from pyspark.sql.functions import *

silver_df = spark.read.table("silver_stock_data")

silver_df.show(5)

+-------------------+------------------+------------------+------------------+------------------+--------+--------------------+--------------+--------------------+--------------------+-----------------+-----------------+----------+-------------------+
|               Date|              Open|              High|               Low|             Close|  Volume| ingestion_timestamp|ingestion_date| processed_timestamp|        daily_return|             ma_7|            ma_30|avg_vol_30|      volatility_30|
+-------------------+------------------+------------------+------------------+------------------+--------+--------------------+--------------+--------------------+--------------------+-----------------+-----------------+----------+-------------------+
|2026-02-02 05:00:00| 259.7869174093499| 270.2371306501278| 258.9676766447768| 269.7575988769531|73913400|2026-03-17 12:09:...|    2026-03-17|2026-03-17 12:11:...|                NULL|269.7575988769531|269.7575988769531| 7.39134E7|             

In [0]:
gold_df = silver_df.withColumn(
    "REQUIRES_COMPLIANCE_REVIEW",
    when(col("Volume") > 3 * col("avg_vol_30"), True).otherwise(False)
)

In [0]:
gold_df = gold_df.withColumn(
    "risk_score",
    abs(col("daily_return")) * col("volatility_30")
)

In [0]:
compliance_summary = gold_df.groupBy().agg(
    sum(col("REQUIRES_COMPLIANCE_REVIEW").cast("int")).alias("total_flags")
)

In [0]:
high_risk_days = gold_df.filter(col("risk_score") > 0.05)

In [0]:
from pyspark.sql.functions import weekofyear

weekly_risk = gold_df.groupBy(
    weekofyear("Date").alias("week")
).agg(
    avg("risk_score").alias("avg_risk_score")
)

In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_stock_risk")

In [0]:
compliance_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_compliance_summary")

high_risk_days.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_high_risk_days")

weekly_risk.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_weekly_risk")

In [0]:
%sql
SELECT * FROM gold_stock_risk LIMIT 5;

Date,Open,High,Low,Close,Volume,ingestion_timestamp,ingestion_date,processed_timestamp,daily_return,ma_7,ma_30,avg_vol_30,volatility_30,REQUIRES_COMPLIANCE_REVIEW,risk_score
2026-02-02T05:00:00.000Z,259.7869174093499,270.2371306501278,258.9676766447768,269.7575988769531,73913400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,null,269.7575988769531,269.7575988769531,7.39134E7,null,false,null
2026-02-03T05:00:00.000Z,268.9483513556569,271.6258386480508,267.3598109321867,269.22808837890625,64394700,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.001962912259937505,269.4928436279297,269.4928436279297,6.915405E7,0.37442046387841144,false,7.349545189184215E-4
2026-02-04T05:00:00.000Z,272.03545112075,278.68922850453845,272.03545112075,276.23150634765625,90545700,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,0.02601295433518634,271.7390645345052,271.7390645345052,7.62846E7,3.8995666971211462,false,0.10143925041922579
2026-02-05T05:00:00.000Z,277.86999494352693,279.2387093202658,272.97458180817284,275.6520690917969,52977400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,-0.0020976508564164785,272.7173156738281,272.7173156738281,7.04578E7,3.7370641038847494,false,0.007839055717997125
2026-02-06T05:00:00.000Z,276.8609202349588,280.6473855638201,276.67109542368036,277.8599853515625,50453400,2026-03-17T12:09:06.093Z,2026-03-17,2026-03-17T12:11:01.325Z,0.00800979389358529,273.745849609375,273.745849609375,6.645692E7,3.970345875394946,false,0.031801652148159984
